# FLARE AUSAM: Adaptive Unified Segmentation Anything Model

Unified reimplementation of the FLARE segmentation pipeline. Each section corresponds to one of the original notebooks:

| Section | Original Notebook | Class | Method |
|---------|------------------|-------|--------|
| 2 | paper1-aggregation | 12 (Duodenum) | Single-stage fine-tune baseline |
| 3 | paper1 | 4 (Pancreas) | E2H / H2E curriculum learning |
| 4 | paper1 | 4 (Pancreas) | Curriculum + augmentations |
| 5 | paper1-aggregation | 1 → 12 | Transfer learning |
| 6 | Tumor | 14 (Tumor) | Best method applied to tumors |
| 7 | paper1-sam2 | 1 (Liver) | SAM2 architecture |
| 8 | Multi-Class0-3d | 0 (all) | 3D volume aggregation |

**Scores to beat:**
- Class 4 (Pancreas): Dice 0.822
- Class 14 (Tumor): Dice 0.839
- Class 0/1 (Liver): Dice 0.941

## Section 1 — Setup & Utilities

Shared code used by all sections: imports, paths, metrics, DBSCAN prompt generation, dataset class, and training helpers.

In [ ]:
import numpy as np
import torch
import torch.distributed as dist
import torch.multiprocessing as mp
import matplotlib.pyplot as plt
import csv
import os
import socket
import time

from monai.losses import DiceLoss
from scipy.stats import entropy as scipy_entropy
from scipy import ndimage
from sklearn.cluster import DBSCAN
from sklearn.model_selection import train_test_split
from skimage.measure import label, regionprops
from skimage import exposure
from torch.amp import autocast, GradScaler
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset, Subset
from torch.utils.data.distributed import DistributedSampler
from tqdm import tqdm
from transformers import SamModel, SamProcessor

torch.set_num_threads(1)
%matplotlib inline

print(f"PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}, GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
FLARE_DATA_DIR = "/scratch/ud3d4/acm_data/FLARE"
OUTPUT_DIR = "/scratch/ud3d4/acm_data/FLARE/runs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

WORLD_SIZE = torch.cuda.device_count()  # 2 GPUs
SCALE = 1024.0 / 256.0  # SAM input is 1024x1024, our images are 256x256

In [ ]:
# ── Metrics ───────────────────────────────────────────────────────────────────

def compute_dice(pred, gt):
    intersection = torch.sum(pred * gt)
    return (2 * intersection + 1e-6) / (torch.sum(pred) + torch.sum(gt) + 1e-6)

def compute_iou(pred, gt):
    inter = torch.logical_and(pred, gt).sum().float()
    union = torch.logical_or(pred, gt).sum().float()
    return inter / (union + 1e-6)

def compute_accuracy(pred, gt):
    return (pred.view(-1) == gt.view(-1)).sum().float() / gt.numel()

def compute_precision(pred, gt):
    tp = torch.sum((pred == 1) & (gt == 1)).float()
    fp = torch.sum((pred == 1) & (gt == 0)).float()
    return tp / (tp + fp + 1e-6)

def compute_sensitivity(pred, gt):
    tp = torch.sum((pred == 1) & (gt == 1)).float()
    fn = torch.sum((pred == 0) & (gt == 1)).float()
    return tp / (tp + fn + 1e-6)

def compute_specificity(pred, gt):
    tn = torch.sum((pred == 0) & (gt == 0)).float()
    fp = torch.sum((pred == 1) & (gt == 0)).float()
    return tn / (tn + fp + 1e-6)

def all_metrics(pred_binary, gt):
    """Compute all metrics at once. Returns dict."""
    return {
        "dice": compute_dice(pred_binary.float(), gt.float()).item(),
        "iou": compute_iou(pred_binary, gt.bool()).item(),
        "acc": compute_accuracy(pred_binary, gt.bool()).item(),
        "precision": compute_precision(pred_binary, gt).item(),
        "sensitivity": compute_sensitivity(pred_binary, gt).item(),
        "specificity": compute_specificity(pred_binary, gt).item(),
    }

In [ ]:
# ── DBSCAN Point Prompt Generation ────────────────────────────────────────────

def apply_dbscan(region_coords, eps=5, min_samples=10):
    """Cluster region pixels with DBSCAN, return cluster centers."""
    db = DBSCAN(eps=eps, min_samples=min_samples).fit(region_coords)
    centers = []
    for k in set(db.labels_):
        if k == -1:
            continue
        centers.append(region_coords[db.labels_ == k].mean(axis=0))
    return np.array(centers) if centers else np.array([])

def generate_coordinates(masks, eps=5, min_samples=10):
    """For each mask, extract point prompts via connected-component + DBSCAN.
    Returns array of (slice_idx, row, col) tuples.
    """
    coordinates = []
    for i, mask in enumerate(masks):
        labeled_mask = label(mask)
        for region in regionprops(labeled_mask):
            region_coords = np.array(region.coords)
            cluster_centers = apply_dbscan(region_coords, eps, min_samples)
            if cluster_centers.size == 0:
                centroid = np.round(region.centroid).astype(int)
                coordinates.append((i, centroid[0], centroid[1]))
            else:
                for center in cluster_centers:
                    coordinates.append((i, int(center[0]), int(center[1])))
    return np.array(coordinates)

In [ ]:
# ── Dataset & Data Loading ────────────────────────────────────────────────────

class FLAREDataset(Dataset):
    def __init__(self, images, labels, coordinates):
        self.images = images
        self.labels = labels
        self.coordinates = coordinates

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]    # (H, W, 3) uint8
        lbl = self.labels[idx]      # (H, W) binary
        coords = [(pt[2], pt[1]) for pt in self.coordinates if pt[0] == idx]
        return image, lbl, coords

def collate_fn(batch):
    images, labels, coords = zip(*batch)
    return list(images), list(labels), list(coords)

def load_and_split(class_id, test_size=0.2, val_size=0.5, seed=42):
    """Load a FLARE class, binarize labels, split into train/val/test."""
    images = np.load(os.path.join(FLARE_DATA_DIR, f"class_{class_id}_images.npy"))
    labels = np.load(os.path.join(FLARE_DATA_DIR, f"class_{class_id}_labels.npy"))
    labels = (labels > 0).astype(np.uint8)  # organ=1, background=0

    x_train, x_temp, y_train, y_temp = train_test_split(images, labels, test_size=test_size, random_state=seed)
    x_val, x_test, y_val, y_test = train_test_split(x_temp, y_temp, test_size=val_size, random_state=seed)

    print(f"Class {class_id}: train={x_train.shape[0]}, val={x_val.shape[0]}, test={x_test.shape[0]}")
    return x_train, y_train, x_val, y_val, x_test, y_test

def get_or_generate_coords(labels, split_name, class_id, eps=5, min_samples=10):
    """Load cached coordinates or generate them."""
    cache = os.path.join(OUTPUT_DIR, f"{split_name}_coords_class{class_id}.npy")
    if os.path.exists(cache):
        return np.load(cache, allow_pickle=True)
    print(f"  Generating {split_name} coordinates...")
    coords = generate_coordinates(labels, eps=eps, min_samples=min_samples)
    np.save(cache, coords)
    return coords

In [ ]:
# ── Training Helpers ──────────────────────────────────────────────────────────

def find_free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("", 0))
        return s.getsockname()[1]

def setup_ddp(rank, world_size, port):
    os.environ["MASTER_ADDR"] = "localhost"
    os.environ["MASTER_PORT"] = str(port)
    dist.init_process_group("nccl", rank=rank, world_size=world_size)

def prepare_batch(images_batch, labels_batch, coords_batch, processor, device):
    """Convert a batch of numpy images/labels/coords into SAM-ready tensors.
    Returns: pixel_values, input_points, input_labels, gt_masks
    """
    # Images: HWC uint8 → CHW float → back to HWC for processor
    imgs_for_processor = []
    for img in images_batch:
        if isinstance(img, np.ndarray):
            img = img.astype(np.float32)
        if img.ndim == 3 and img.shape[-1] == 3:
            imgs_for_processor.append(img)
        elif img.ndim == 2:
            imgs_for_processor.append(np.stack([img]*3, axis=-1))

    inputs = processor(images=imgs_for_processor, return_tensors="pt", do_rescale=False)
    pixel_values = inputs["pixel_values"].to(device)

    # Point prompts: (B, 1, max_pts, 2) and labels (B, 1, max_pts)
    pts_list, lbs_list = [], []
    for coords in coords_batch:
        if len(coords) > 0:
            pts = torch.tensor(coords, dtype=torch.float32) * SCALE
            pts_list.append(pts.unsqueeze(0))
            lbs_list.append(torch.ones(1, len(coords), dtype=torch.long))
        else:
            pts_list.append(torch.tensor([[[512.0, 512.0]]]))
            lbs_list.append(torch.ones(1, 1, dtype=torch.long))

    max_pts = max(p.shape[1] for p in pts_list)
    padded_pts, padded_lbs = [], []
    for pts, lbs in zip(pts_list, lbs_list):
        n = pts.shape[1]
        if n < max_pts:
            pts = torch.cat([pts, torch.zeros(1, max_pts - n, 2)], dim=1)
            lbs = torch.cat([lbs, -torch.ones(1, max_pts - n, dtype=torch.long)], dim=1)
        padded_pts.append(pts)
        padded_lbs.append(lbs)

    input_points = torch.cat(padded_pts, dim=0).unsqueeze(1).to(device)  # (B, 1, max_pts, 2)
    input_labels = torch.cat(padded_lbs, dim=0).unsqueeze(1).to(device)  # (B, 1, max_pts)

    # Ground truth masks
    gt_masks = []
    for lbl in labels_batch:
        if isinstance(lbl, np.ndarray):
            lbl = torch.from_numpy(lbl).float()
        gt_masks.append(lbl.unsqueeze(0))
    gt_masks = torch.stack(gt_masks).to(device)  # (B, 1, H, W)

    return pixel_values, input_points, input_labels, gt_masks

def forward_sam(sam, pixel_values, input_points, input_labels, gt_masks, loss_fn):
    """Run SAM forward pass, return loss and pred_binary."""
    outputs = sam(
        pixel_values=pixel_values,
        input_points=input_points,
        input_labels=input_labels,
        multimask_output=False,
    )
    pred_probs = torch.sigmoid(outputs.pred_masks.squeeze(1))
    if pred_probs.shape != gt_masks.shape:
        gt_masks = torch.nn.functional.interpolate(gt_masks, size=pred_probs.shape[-2:], mode="nearest")
    loss = loss_fn(pred_probs, gt_masks)
    pred_binary = (pred_probs > 0.5).float()
    return loss, pred_binary, gt_masks

In [ ]:
# ── Generic DDP Training & Test Evaluation ────────────────────────────────────

def train_worker(rank, world_size, port, cfg, x_train, y_train, x_val, y_val, 
                 train_coords, val_coords, train_subset_indices=None):
    """Generic DDP training worker. cfg is a dict with hyperparameters."""
    setup_ddp(rank, world_size, port)
    device = torch.device(f"cuda:{rank}")

    processor = SamProcessor.from_pretrained("facebook/sam-vit-base")
    sam = SamModel.from_pretrained("facebook/sam-vit-base").to(device)

    # Optional: load pretrained checkpoint for transfer learning
    if cfg.get("pretrained_path") and os.path.exists(cfg["pretrained_path"]):
        state = torch.load(cfg["pretrained_path"], map_location=device)
        cleaned = {k.replace("module.", ""): v for k, v in state.items()}
        sam.load_state_dict(cleaned, strict=False)
        if rank == 0:
            print(f"  Loaded pretrained weights from {cfg['pretrained_path']}")

    sam = DDP(sam, device_ids=[rank], output_device=rank, find_unused_parameters=True)
    loss_fn = DiceLoss(to_onehot_y=False, sigmoid=False)
    optimizer = Adam(sam.parameters(), lr=cfg.get("lr", 1e-5))
    scaler = GradScaler("cuda")

    train_dataset = FLAREDataset(x_train, y_train, train_coords)
    val_dataset = FLAREDataset(x_val, y_val, val_coords)

    best_val_loss = float("inf")
    no_improve = 0
    batch_size = cfg.get("batch_size", 5)
    patience = cfg.get("patience", 10)
    model_save_path = cfg["model_save_path"]
    csv_path = cfg.get("csv_path", model_save_path.replace(".pth", "_metrics.csv"))

    if rank == 0:
        with open(csv_path, "w", newline="") as f:
            csv.writer(f).writerow(["Epoch","TrLoss","VaLoss","TrDice","VaDice","TrIoU","VaIoU","PctData"])

    for epoch in range(cfg.get("epochs", 250)):
        # Determine training subset (for curriculum learning)
        if train_subset_indices is not None:
            subset = Subset(train_dataset, train_subset_indices)
        else:
            subset = train_dataset

        train_sampler = DistributedSampler(subset, num_replicas=world_size, rank=rank, shuffle=True)
        train_sampler.set_epoch(epoch)
        train_loader = DataLoader(subset, batch_size=batch_size, sampler=train_sampler, collate_fn=collate_fn)

        # ── Train ──
        sam.train()
        ep = {"loss": [], "dice": [], "iou": []}
        for images_b, labels_b, coords_b in train_loader:
            pixel_values, input_points, input_labels, gt_masks = prepare_batch(
                images_b, labels_b, coords_b, processor, device)
            optimizer.zero_grad()
            with autocast("cuda"):
                loss, pred_binary, gt_masks = forward_sam(sam, pixel_values, input_points, input_labels, gt_masks, loss_fn)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            ep["loss"].append(loss.item())
            ep["dice"].append(compute_dice(pred_binary, gt_masks).item())
            ep["iou"].append(compute_iou(pred_binary, gt_masks.bool()).item())

        dist.barrier()

        # ── Validate ──
        val_sampler = DistributedSampler(val_dataset, num_replicas=world_size, rank=rank)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, sampler=val_sampler, collate_fn=collate_fn)
        sam.eval()
        ev = {"loss": [], "dice": [], "iou": []}
        with torch.no_grad():
            for images_b, labels_b, coords_b in val_loader:
                pixel_values, input_points, input_labels, gt_masks = prepare_batch(
                    images_b, labels_b, coords_b, processor, device)
                with autocast("cuda"):
                    loss, pred_binary, gt_masks = forward_sam(sam, pixel_values, input_points, input_labels, gt_masks, loss_fn)
                ev["loss"].append(loss.item())
                ev["dice"].append(compute_dice(pred_binary, gt_masks).item())
                ev["iou"].append(compute_iou(pred_binary, gt_masks.bool()).item())

        avg_tr = {k: np.mean(v) for k, v in ep.items()}
        avg_va = {k: np.mean(v) for k, v in ev.items()}
        pct = len(subset) / len(train_dataset) * 100

        if rank == 0:
            print(f"  E{epoch+1}: TrDice={avg_tr['dice']:.4f} VaDice={avg_va['dice']:.4f} "
                  f"VaLoss={avg_va['loss']:.4f} Data={pct:.0f}%")
            with open(csv_path, "a", newline="") as f:
                csv.writer(f).writerow([epoch+1, avg_tr["loss"], avg_va["loss"],
                                        avg_tr["dice"], avg_va["dice"], avg_tr["iou"], avg_va["iou"], pct])

        if avg_va["loss"] < best_val_loss:
            best_val_loss = avg_va["loss"]
            no_improve = 0
            if rank == 0:
                torch.save(sam.state_dict(), model_save_path)
        else:
            no_improve += 1
            if no_improve >= patience:
                if rank == 0:
                    print(f"  Early stopping at epoch {epoch+1}")
                break
        dist.barrier()

    dist.destroy_process_group()


def evaluate_test(model_path, x_test, y_test, test_coords):
    """Single-GPU test evaluation. Returns dict of mean metrics."""
    device = torch.device("cuda:0")
    processor = SamProcessor.from_pretrained("facebook/sam-vit-base")
    sam = SamModel.from_pretrained("facebook/sam-vit-base")

    assert os.path.exists(model_path), f"No checkpoint at {model_path}"
    state = torch.load(model_path, map_location=device)
    cleaned = {k.replace("module.", ""): v for k, v in state.items()}
    sam.load_state_dict(cleaned, strict=False)
    sam.to(device).eval()

    test_dataset = FLAREDataset(x_test, y_test, test_coords)
    loader = DataLoader(test_dataset, batch_size=1, collate_fn=collate_fn)

    results = {k: [] for k in ["dice", "iou", "acc", "precision", "sensitivity", "specificity"]}

    with torch.no_grad():
        for images_b, labels_b, coords_b in tqdm(loader, desc="Testing"):
            pixel_values, input_points, input_labels, gt_masks = prepare_batch(
                images_b, labels_b, coords_b, processor, device)
            outputs = sam(pixel_values=pixel_values, input_points=input_points,
                         input_labels=input_labels, multimask_output=False)
            pred_probs = torch.sigmoid(outputs.pred_masks.squeeze(1))
            if pred_probs.shape != gt_masks.shape:
                gt_masks = torch.nn.functional.interpolate(gt_masks, size=pred_probs.shape[-2:], mode="nearest")
            pred_binary = (pred_probs > 0.5).float()
            m = all_metrics(pred_binary, gt_masks)
            for k in results:
                results[k].append(m[k])

    print("\n=== Test Results ===")
    for k, v in results.items():
        print(f"  {k}: {np.mean(v):.4f} +/- {np.std(v):.4f}")
    return {k: (np.mean(v), np.std(v)) for k, v in results.items()}


def run_training(cfg, x_train, y_train, x_val, y_val, train_coords, val_coords, 
                 train_subset_indices=None):
    """Launch DDP training across all GPUs."""
    port = find_free_port()
    mp.spawn(
        train_worker,
        args=(WORLD_SIZE, port, cfg, x_train, y_train, x_val, y_val,
              train_coords, val_coords, train_subset_indices),
        nprocs=WORLD_SIZE, join=True,
    )

## Section 2 — Single-Stage Baseline (paper1-aggregation)

Simple fine-tune of SAM on class 12 (Duodenum) from pretrained SAM base weights. No curriculum, no transfer learning. This establishes the baseline to beat.

**Original notebook**: `SAM-DBSCAN_FLARE-paper1-aggregation.ipynb` (Fine tune - Single Stage cell)

In [ ]:
# Load class 12 data
x_train_12, y_train_12, x_val_12, y_val_12, x_test_12, y_test_12 = load_and_split(12)
train_coords_12 = get_or_generate_coords(y_train_12, "train", 12)
val_coords_12 = get_or_generate_coords(y_val_12, "val", 12)
test_coords_12 = get_or_generate_coords(y_test_12, "test", 12)
print(f"Coordinates: train={len(train_coords_12)}, val={len(val_coords_12)}, test={len(test_coords_12)}")

In [ ]:
# Train single-stage baseline on class 12
cfg_baseline = {
    "model_save_path": os.path.join(OUTPUT_DIR, "s2_baseline_class12.pth"),
    "epochs": 250, "batch_size": 5, "lr": 1e-5, "patience": 10,
}
run_training(cfg_baseline, x_train_12, y_train_12, x_val_12, y_val_12, train_coords_12, val_coords_12)

In [ ]:
# Evaluate baseline on test set
results_s2 = evaluate_test(cfg_baseline["model_save_path"], x_test_12, y_test_12, test_coords_12)

## Section 3 — Curriculum Learning: E2H & H2E (paper1)

The core AUSAM method. Instead of training on all data from epoch 1, samples are ranked by entropy (image complexity) and introduced progressively:

- **E2H (Easy-to-Hard)**: start with low-entropy (easy) samples, progressively add high-entropy (hard) ones
- **H2E (Hard-to-Easy)**: start with high-entropy (hard) samples, progressively add low-entropy (easy) ones

The entropy of each image determines its difficulty. The training data percentage grows when validation loss plateaus, controlled by entropy derivative thresholds.

**Original notebook**: `SAM-DBSCAN_FLARE-paper1.ipynb` (E2H and H2E cells)  
**Target**: Class 4 (Pancreas) Dice > 0.822

In [ ]:
# ── Entropy-based Curriculum Helpers ──────────────────────────────────────────

def calculate_entropy(image):
    """Compute Shannon entropy of an image's pixel histogram."""
    flat = image.flatten()
    hist, _ = np.histogram(flat, bins=np.arange(flat.max() + 2))
    hist = hist / hist.sum()
    return scipy_entropy(hist)

def rank_by_entropy(images, order="e2h"):
    """Rank image indices by entropy. 
    e2h = easy-to-hard (ascending entropy), h2e = hard-to-easy (descending).
    """
    entropies = np.array([calculate_entropy(img) for img in images])
    if order == "e2h":
        return np.argsort(entropies), entropies  # low entropy first
    else:  # h2e
        return np.argsort(entropies)[::-1].copy(), entropies  # high entropy first

def determine_initial_percentage(entropies, k=1):
    """Use entropy derivative to find the initial % of training data.
    Returns (initial_pct, entropy_derivative, threshold, mean_deriv, std_deriv).
    """
    entropy_deriv = np.diff(entropies)
    mean_d = np.mean(entropy_deriv)
    std_d = np.std(entropy_deriv)
    threshold = mean_d - k * std_d
    significant = np.where(entropy_deriv < threshold)[0]
    initial_pct = max(10.0, len(significant) / len(entropies) * 100)
    return initial_pct, entropy_deriv, threshold, mean_d, std_d

In [ ]:
# ── Curriculum DDP Training Worker ────────────────────────────────────────────

def curriculum_train_worker(rank, world_size, port, cfg, x_train, y_train, x_val, y_val,
                            train_coords, val_coords, sorted_indices, entropies):
    """DDP worker with curriculum learning: progressively adds data."""
    setup_ddp(rank, world_size, port)
    device = torch.device(f"cuda:{rank}")

    processor = SamProcessor.from_pretrained("facebook/sam-vit-base")
    sam = SamModel.from_pretrained("facebook/sam-vit-base").to(device)
    if cfg.get("pretrained_path") and os.path.exists(cfg["pretrained_path"]):
        state = torch.load(cfg["pretrained_path"], map_location=device)
        cleaned = {k.replace("module.", ""): v for k, v in state.items()}
        sam.load_state_dict(cleaned, strict=False)

    sam = DDP(sam, device_ids=[rank], output_device=rank, find_unused_parameters=True)
    loss_fn = DiceLoss(to_onehot_y=False, sigmoid=False)
    optimizer = Adam(sam.parameters(), lr=cfg.get("lr", 1e-5))
    scaler = GradScaler("cuda")

    train_dataset = FLAREDataset(x_train, y_train, train_coords)
    val_dataset = FLAREDataset(x_val, y_val, val_coords)

    # Curriculum state
    sorted_entropies = entropies[sorted_indices]
    init_pct, entropy_deriv, threshold, mean_d, std_d = determine_initial_percentage(sorted_entropies)
    pct = init_pct
    k = 1.0
    data_increment_patience = 5

    best_val_loss = float("inf")
    no_improve = 0
    batch_size = cfg.get("batch_size", 5)
    patience = cfg.get("patience", 10)
    model_save_path = cfg["model_save_path"]
    csv_path = model_save_path.replace(".pth", "_metrics.csv")

    if rank == 0:
        print(f"  Initial data percentage: {pct:.1f}%")
        with open(csv_path, "w", newline="") as f:
            csv.writer(f).writerow(["Epoch","TrLoss","VaLoss","TrDice","VaDice","TrIoU","VaIoU","PctData","k"])

    for epoch in range(cfg.get("epochs", 1000)):
        # Select curriculum subset
        n_samples = max(1, int(len(sorted_indices) * pct / 100.0))
        subset_indices = sorted_indices[:n_samples].tolist()
        subset = Subset(train_dataset, subset_indices)

        train_sampler = DistributedSampler(subset, num_replicas=world_size, rank=rank, shuffle=True)
        train_sampler.set_epoch(epoch)
        train_loader = DataLoader(subset, batch_size=batch_size, sampler=train_sampler, collate_fn=collate_fn)

        # ── Train ──
        sam.train()
        ep = {"loss": [], "dice": [], "iou": []}
        for images_b, labels_b, coords_b in train_loader:
            pv, ip, il, gt = prepare_batch(images_b, labels_b, coords_b, processor, device)
            optimizer.zero_grad()
            with autocast("cuda"):
                loss, pred_bin, gt = forward_sam(sam, pv, ip, il, gt, loss_fn)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(sam.parameters(), max_norm=2.0)
            scaler.step(optimizer)
            scaler.update()
            ep["loss"].append(loss.item())
            ep["dice"].append(compute_dice(pred_bin, gt).item())
            ep["iou"].append(compute_iou(pred_bin, gt.bool()).item())

        dist.barrier()

        # ── Validate ──
        val_sampler = DistributedSampler(val_dataset, num_replicas=world_size, rank=rank)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, sampler=val_sampler, collate_fn=collate_fn)
        sam.eval()
        ev = {"loss": [], "dice": [], "iou": []}
        with torch.no_grad():
            for images_b, labels_b, coords_b in val_loader:
                pv, ip, il, gt = prepare_batch(images_b, labels_b, coords_b, processor, device)
                with autocast("cuda"):
                    loss, pred_bin, gt = forward_sam(sam, pv, ip, il, gt, loss_fn)
                ev["loss"].append(loss.item())
                ev["dice"].append(compute_dice(pred_bin, gt).item())
                ev["iou"].append(compute_iou(pred_bin, gt.bool()).item())

        avg_tr = {m: np.mean(v) for m, v in ep.items()}
        avg_va = {m: np.mean(v) for m, v in ev.items()}

        if rank == 0:
            print(f"  E{epoch+1}: TrDice={avg_tr['dice']:.4f} VaDice={avg_va['dice']:.4f} "
                  f"VaLoss={avg_va['loss']:.4f} Data={pct:.1f}% k={k:.3f}")
            with open(csv_path, "a", newline="") as f:
                csv.writer(f).writerow([epoch+1, avg_tr["loss"], avg_va["loss"],
                                        avg_tr["dice"], avg_va["dice"], avg_tr["iou"], avg_va["iou"], pct, k])

        if avg_va["loss"] < best_val_loss:
            best_val_loss = avg_va["loss"]
            no_improve = 0
            if rank == 0:
                torch.save(sam.state_dict(), model_save_path)
        else:
            no_improve += 1
            if no_improve >= patience:
                if rank == 0:
                    print(f"  Early stopping at epoch {epoch+1}")
                break
            # Expand data when stuck
            if no_improve % data_increment_patience == 0 and pct < 100:
                old_pct = pct
                k /= 2
                threshold = mean_d - k * std_d
                significant = np.where(entropy_deriv < threshold)[0]
                pct = min(100.0, max(pct + 10, len(significant) / len(sorted_indices) * 100))
                if rank == 0 and pct > old_pct:
                    print(f"  >> Expanding data: {old_pct:.1f}% -> {pct:.1f}% (k={k:.3f})")

        dist.barrier()

    dist.destroy_process_group()


def run_curriculum_training(cfg, x_train, y_train, x_val, y_val, 
                            train_coords, val_coords, order="e2h"):
    """Launch curriculum DDP training."""
    sorted_indices, entropies = rank_by_entropy(x_train, order=order)
    port = find_free_port()
    mp.spawn(
        curriculum_train_worker,
        args=(WORLD_SIZE, port, cfg, x_train, y_train, x_val, y_val,
              train_coords, val_coords, sorted_indices, entropies),
        nprocs=WORLD_SIZE, join=True,
    )

In [ ]:
# Load class 4 (Pancreas) data
x_train_4, y_train_4, x_val_4, y_val_4, x_test_4, y_test_4 = load_and_split(4)
train_coords_4 = get_or_generate_coords(y_train_4, "train", 4)
val_coords_4 = get_or_generate_coords(y_val_4, "val", 4)
test_coords_4 = get_or_generate_coords(y_test_4, "test", 4)

In [ ]:
# E2H curriculum on class 4 (Pancreas) — target: beat Dice 0.822
cfg_e2h = {
    "model_save_path": os.path.join(OUTPUT_DIR, "s3_e2h_class4.pth"),
    "epochs": 1000, "batch_size": 5, "lr": 1e-5, "patience": 10,
}
print("=== E2H (Easy-to-Hard) on Class 4 ===")
run_curriculum_training(cfg_e2h, x_train_4, y_train_4, x_val_4, y_val_4,
                        train_coords_4, val_coords_4, order="e2h")

In [ ]:
# H2E curriculum on class 4 (Pancreas)
cfg_h2e = {
    "model_save_path": os.path.join(OUTPUT_DIR, "s3_h2e_class4.pth"),
    "epochs": 1000, "batch_size": 5, "lr": 1e-5, "patience": 10,
}
print("=== H2E (Hard-to-Easy) on Class 4 ===")
run_curriculum_training(cfg_h2e, x_train_4, y_train_4, x_val_4, y_val_4,
                        train_coords_4, val_coords_4, order="h2e")

In [ ]:
# Evaluate both curriculum methods
print("=== E2H Test Results ===")
results_s3_e2h = evaluate_test(cfg_e2h["model_save_path"], x_test_4, y_test_4, test_coords_4)
print("\n=== H2E Test Results ===")
results_s3_h2e = evaluate_test(cfg_h2e["model_save_path"], x_test_4, y_test_4, test_coords_4)

## Section 4 — Curriculum + Augmentations (paper1)

Adds data augmentation on top of curriculum learning. Two augmentations from the original notebook:
- **Intensity rescaling**: percentile-based contrast enhancement
- **Gaussian smoothing**: slight blur to reduce noise

**Original notebook**: `SAM-DBSCAN_FLARE-paper1.ipynb` (H2E Augmentations cell)

In [ ]:
# ── SAM Augmenter ─────────────────────────────────────────────────────────────

import random

class SAMAugmenter:
    """Augmentations from the original H2E Augmentations notebook cell."""
    def __init__(self, gaussian_sigma=0.5):
        self.gaussian_sigma = gaussian_sigma

    def rescale(self, image):
        p2, p98 = np.percentile(image, (2, 98))
        return exposure.rescale_intensity(image, in_range=(p2, p98))

    def gaussian(self, image):
        return ndimage.gaussian_filter(image, self.gaussian_sigma)

    def augment(self, image, mask, points):
        aug_type = random.choice(["rescale", "gaussian"])
        if aug_type == "rescale":
            return self.rescale(image), mask, points
        else:
            return self.gaussian(image), mask, points


def augment_dataset(images, labels, coordinates, augmenter, num_augmentations=5):
    """Generate augmented copies of the dataset."""
    aug_images, aug_labels = list(images), list(labels)
    aug_coords = list(coordinates)

    # Build per-index coordinate lookup
    idx_coords = {}
    for pt in coordinates:
        idx_coords.setdefault(pt[0], []).append(pt)

    new_idx = len(images)
    for idx in range(len(images)):
        pts = np.array(idx_coords.get(idx, []))
        for _ in range(num_augmentations):
            aug_img, aug_lbl, _ = augmenter.augment(images[idx], labels[idx], pts)
            aug_images.append(aug_img)
            aug_labels.append(aug_lbl)
            # Remap coordinates to new index
            for pt in idx_coords.get(idx, []):
                aug_coords.append((new_idx, pt[1], pt[2]))
            new_idx += 1

    return np.array(aug_images), np.array(aug_labels), np.array(aug_coords)

In [ ]:
# H2E + Augmentations on class 4
augmenter = SAMAugmenter()
print("Augmenting training data...")
aug_images_4, aug_labels_4, aug_coords_4 = augment_dataset(
    x_train_4, y_train_4, train_coords_4, augmenter, num_augmentations=2)
print(f"  Original: {len(x_train_4)}, Augmented: {len(aug_images_4)}")

cfg_h2e_aug = {
    "model_save_path": os.path.join(OUTPUT_DIR, "s4_h2e_aug_class4.pth"),
    "epochs": 1000, "batch_size": 5, "lr": 1e-5, "patience": 10,
}
print("=== H2E + Augmentations on Class 4 ===")
run_curriculum_training(cfg_h2e_aug, aug_images_4, aug_labels_4, x_val_4, y_val_4,
                        aug_coords_4, val_coords_4, order="h2e")

In [ ]:
print("=== H2E + Augmentations Test Results ===")
results_s4 = evaluate_test(cfg_h2e_aug["model_save_path"], x_test_4, y_test_4, test_coords_4)

## Section 5 — Transfer Learning (paper1-aggregation)

Train on class 1 (Liver, large/easy), then fine-tune on class 12 (Duodenum, small/hard). Compare against Section 2 baseline (class 12 from scratch).

**Original notebook**: `SAM-DBSCAN_FLARE-paper1-aggregation.ipynb`

In [ ]:
# Step 1: Train on class 1 (Liver)
x_train_1, y_train_1, x_val_1, y_val_1, x_test_1, y_test_1 = load_and_split(1)
train_coords_1 = get_or_generate_coords(y_train_1, "train", 1)
val_coords_1 = get_or_generate_coords(y_val_1, "val", 1)
test_coords_1 = get_or_generate_coords(y_test_1, "test", 1)

cfg_liver = {
    "model_save_path": os.path.join(OUTPUT_DIR, "s5_liver_class1.pth"),
    "epochs": 250, "batch_size": 5, "lr": 1e-5, "patience": 10,
}
print("=== Training on Class 1 (Liver) ===")
run_training(cfg_liver, x_train_1, y_train_1, x_val_1, y_val_1, train_coords_1, val_coords_1)

In [ ]:
# Step 2: Fine-tune on class 12 (Duodenum) using Liver weights
cfg_transfer = {
    "model_save_path": os.path.join(OUTPUT_DIR, "s5_transfer_class12.pth"),
    "pretrained_path": os.path.join(OUTPUT_DIR, "s5_liver_class1.pth"),
    "epochs": 250, "batch_size": 5, "lr": 1e-5, "patience": 10,
}
print("=== Transfer Learning: Liver -> Duodenum ===")
run_training(cfg_transfer, x_train_12, y_train_12, x_val_12, y_val_12, train_coords_12, val_coords_12)

In [ ]:
# Evaluate transfer learning vs baseline
print("=== Transfer Learning Test Results (Class 12) ===")
results_s5 = evaluate_test(cfg_transfer["model_save_path"], x_test_12, y_test_12, test_coords_12)

## Section 6 — Tumor Segmentation (Tumor notebook)

Apply the best method from Sections 3-4 to class 14 (Tumor). Tumors are irregular and variable — harder than organs.

**Original notebook**: `SAM-DBSCAN_FLARE-Tumor.ipynb`  
**Target**: Dice > 0.839

In [ ]:
# Load class 14 (Tumor) data
x_train_14, y_train_14, x_val_14, y_val_14, x_test_14, y_test_14 = load_and_split(14)
train_coords_14 = get_or_generate_coords(y_train_14, "train", 14)
val_coords_14 = get_or_generate_coords(y_val_14, "val", 14)
test_coords_14 = get_or_generate_coords(y_test_14, "test", 14)

# Use best curriculum method (H2E based on paper1 results)
cfg_tumor = {
    "model_save_path": os.path.join(OUTPUT_DIR, "s6_h2e_class14.pth"),
    "epochs": 1000, "batch_size": 5, "lr": 1e-5, "patience": 10,
}
print("=== H2E on Class 14 (Tumor) ===")
run_curriculum_training(cfg_tumor, x_train_14, y_train_14, x_val_14, y_val_14,
                        train_coords_14, val_coords_14, order="h2e")

In [ ]:
print("=== Tumor Test Results ===")
results_s6 = evaluate_test(cfg_tumor["model_save_path"], x_test_14, y_test_14, test_coords_14)

## Section 7 — SAM2 (paper1-sam2)

Replace SAM ViT-Base with SAM2 Hiera architecture. SAM2 uses a hierarchical vision transformer which may capture multi-scale features better for medical imaging.

**Original notebook**: `SAM-DBSCAN_FLARE-paper1-sam2.ipynb`  
**Target**: Dice > 0.941 (Class 1, Liver)

**Note**: Requires `sam2` package. Install with: `pip install sam2`

In [ ]:
# ── SAM2 Training (single-GPU, per-image like the original notebook) ──────────

try:
    from sam2.sam2_image_predictor import SAM2ImagePredictor
    SAM2_AVAILABLE = True
except ImportError:
    SAM2_AVAILABLE = False
    print("SAM2 not installed. Run: pip install sam2")


def train_sam2(cfg, x_train, y_train, x_val, y_val, train_coords, val_coords):
    """Train SAM2 on a single GPU (SAM2 doesn't use the same API as SAM1)."""
    if not SAM2_AVAILABLE:
        print("Skipping SAM2 — not installed.")
        return

    device = torch.device("cuda:0")
    predictor = SAM2ImagePredictor.from_pretrained("facebook/sam2-hiera-base-plus")
    predictor.model.to(device)

    # Unfreeze all parameters
    for param in predictor.model.parameters():
        param.requires_grad = True

    optimizer = Adam(predictor.model.parameters(), lr=cfg.get("lr", 1e-5))
    loss_fn = DiceLoss()

    train_dataset = FLAREDataset(x_train, y_train, train_coords)
    val_dataset = FLAREDataset(x_val, y_val, val_coords)
    train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

    best_val_loss = float("inf")
    no_improve = 0
    model_save_path = cfg["model_save_path"]
    csv_path = model_save_path.replace(".pth", "_metrics.csv")

    with open(csv_path, "w", newline="") as f:
        csv.writer(f).writerow(["Epoch", "TrLoss", "VaLoss", "TrDice", "VaDice"])

    for epoch in range(cfg.get("epochs", 250)):
        # ── Train ──
        predictor.model.train()
        ep_tr = {"loss": [], "dice": []}
        for images_b, labels_b, coords_b in tqdm(train_loader, desc=f"SAM2 Train E{epoch+1}", leave=False):
            image = images_b[0].astype(np.float32)
            lbl = torch.from_numpy(labels_b[0]).float().unsqueeze(0).unsqueeze(0).to(device)
            coords = coords_b[0]

            predictor.set_image(image)
            if len(coords) > 0:
                pts = np.array(coords, dtype=np.float32)
                point_labels = np.ones(len(coords), dtype=np.int32)
            else:
                pts = np.array([[128, 128]], dtype=np.float32)
                point_labels = np.array([1], dtype=np.int32)

            optimizer.zero_grad()
            masks, _, _ = predictor.predict(
                point_coords=pts, point_labels=point_labels, multimask_output=False)
            pred = torch.from_numpy(masks).float().to(device).unsqueeze(1)
            loss = loss_fn(pred, lbl)
            loss.backward()
            optimizer.step()

            pred_bin = (pred > 0.5).float()
            ep_tr["loss"].append(loss.item())
            ep_tr["dice"].append(compute_dice(pred_bin, lbl).item())

        # ── Validate ──
        predictor.model.eval()
        ep_va = {"loss": [], "dice": []}
        with torch.no_grad():
            for images_b, labels_b, coords_b in val_loader:
                image = images_b[0].astype(np.float32)
                lbl = torch.from_numpy(labels_b[0]).float().unsqueeze(0).unsqueeze(0).to(device)
                coords = coords_b[0]

                predictor.set_image(image)
                if len(coords) > 0:
                    pts = np.array(coords, dtype=np.float32)
                    point_labels = np.ones(len(coords), dtype=np.int32)
                else:
                    pts = np.array([[128, 128]], dtype=np.float32)
                    point_labels = np.array([1], dtype=np.int32)

                masks, _, _ = predictor.predict(
                    point_coords=pts, point_labels=point_labels, multimask_output=False)
                pred = torch.from_numpy(masks).float().to(device).unsqueeze(1)
                loss = loss_fn(pred, lbl)
                ep_va["loss"].append(loss.item())
                ep_va["dice"].append(compute_dice((pred > 0.5).float(), lbl).item())

        avg_tr = {k: np.mean(v) for k, v in ep_tr.items()}
        avg_va = {k: np.mean(v) for k, v in ep_va.items()}
        print(f"  E{epoch+1}: TrDice={avg_tr['dice']:.4f} VaDice={avg_va['dice']:.4f} VaLoss={avg_va['loss']:.4f}")

        with open(csv_path, "a", newline="") as f:
            csv.writer(f).writerow([epoch+1, avg_tr["loss"], avg_va["loss"], avg_tr["dice"], avg_va["dice"]])

        if avg_va["loss"] < best_val_loss:
            best_val_loss = avg_va["loss"]
            no_improve = 0
            torch.save(predictor.model.state_dict(), model_save_path)
        else:
            no_improve += 1
            if no_improve >= cfg.get("patience", 10):
                print(f"  Early stopping at epoch {epoch+1}")
                break

In [ ]:
# Train SAM2 on class 1 (Liver) — same class as original sam2 notebook
cfg_sam2 = {
    "model_save_path": os.path.join(OUTPUT_DIR, "s7_sam2_class1.pth"),
    "epochs": 250, "batch_size": 1, "lr": 1e-5, "patience": 10,
}

# Reuse class 1 data from Section 5 (already loaded)
print("=== SAM2 on Class 1 (Liver) ===")
train_sam2(cfg_sam2, x_train_1, y_train_1, x_val_1, y_val_1, train_coords_1, val_coords_1)

## Section 8 — 3D Volume Aggregation (Multi-Class0-3d)

The previous sections evaluate per-slice (2D). This section reconstructs 3D volumes from the raw CT data and evaluates per-subject 3D Dice scores.

**Original notebook**: `SAM-DBSCAN_FLARE-Multi-Class0-paper1-aggregation-one-class-3d.ipynb`

In [ ]:
# ── 3D Evaluation ─────────────────────────────────────────────────────────────
# The original notebook loads per-subject NIfTI volumes, runs SAM slice-by-slice,
# then stacks predictions into a 3D volume and computes volumetric Dice.
# 
# Since the raw NIfTI data was removed (we only have pre-sliced .npy files),
# this section requires the original FLARE NIfTI data to be available.
# We provide the framework — plug in your NIfTI paths to run.

def evaluate_3d_volume(model_path, volume_slices, label_slices, coords_per_slice):
    """Run SAM on each slice of a 3D volume, stack results, compute 3D Dice.
    
    Args:
        model_path: path to trained .pth checkpoint
        volume_slices: list of (H, W, 3) images for one subject
        label_slices: list of (H, W) binary masks for one subject  
        coords_per_slice: list of [(x,y), ...] point prompts per slice
    
    Returns: 3D Dice score
    """
    device = torch.device("cuda:0")
    processor = SamProcessor.from_pretrained("facebook/sam-vit-base")
    sam = SamModel.from_pretrained("facebook/sam-vit-base")
    state = torch.load(model_path, map_location=device)
    cleaned = {k.replace("module.", ""): v for k, v in state.items()}
    sam.load_state_dict(cleaned, strict=False)
    sam.to(device).eval()

    pred_volume = []
    gt_volume = []

    with torch.no_grad():
        for img, lbl, coords in zip(volume_slices, label_slices, coords_per_slice):
            pv, ip, il, gt = prepare_batch([img], [lbl], [coords], processor, device)
            outputs = sam(pixel_values=pv, input_points=ip, input_labels=il, multimask_output=False)
            pred = torch.sigmoid(outputs.pred_masks.squeeze())
            pred_bin = (pred > 0.5).float().cpu().numpy()

            # Resize pred to match GT if needed
            if pred_bin.shape != lbl.shape:
                from skimage.transform import resize
                pred_bin = resize(pred_bin, lbl.shape, order=0, preserve_range=True)

            pred_volume.append(pred_bin)
            gt_volume.append(lbl)

    pred_3d = np.stack(pred_volume)
    gt_3d = np.stack(gt_volume)

    # 3D Dice
    intersection = np.sum(pred_3d * gt_3d)
    dice_3d = (2 * intersection + 1e-6) / (np.sum(pred_3d) + np.sum(gt_3d) + 1e-6)
    return dice_3d, pred_3d

print("3D evaluation framework ready. Requires per-subject NIfTI data to run.")

## Section 9 — Results Summary

In [ ]:
# ── Compile all results ───────────────────────────────────────────────────────

def fmt(result_dict, metric="dice"):
    """Format mean +/- std from results dict."""
    if result_dict and metric in result_dict:
        m, s = result_dict[metric]
        return f"{m:.4f} +/- {s:.4f}"
    return "N/A"

print("=" * 80)
print("FLARE AUSAM — Results Summary")
print("=" * 80)
print(f"{'Section':<10} {'Method':<30} {'Class':<15} {'Test Dice':<20} {'Target'}")
print("-" * 80)

rows = [
    ("S2", "Single-stage baseline",     "12 (Duodenum)", results_s2,      "—"),
    ("S3-E2H", "E2H curriculum",         "4 (Pancreas)",  results_s3_e2h,  "0.822"),
    ("S3-H2E", "H2E curriculum",         "4 (Pancreas)",  results_s3_h2e,  "0.822"),
    ("S4", "H2E + augmentations",        "4 (Pancreas)",  results_s4,      "0.822"),
    ("S5", "Transfer (Liver->Duodenum)", "12 (Duodenum)", results_s5,      "—"),
    ("S6", "H2E tumor",                  "14 (Tumor)",    results_s6,      "0.839"),
]

for sec, method, cls, res, target in rows:
    print(f"{sec:<10} {method:<30} {cls:<15} {fmt(res):<20} {target}")

print("=" * 80)